# 🧩 Andamiaje · Proyecto **Centinela** — Fase 2 (Sistema multimodal especializado)
### Redes Neuronales — Deep Learning · Maestría en Ciencia de Datos · Universidad Santo Tomás

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JotaMao1985/Deep_Learning_Usta/blob/main/notebooks/04-scaffold-centinela-fase2.ipynb)

Este cuaderno es un **andamiaje guía** (*scaffold*) para construir las **tres ramas** de la Fase 2 del Proyecto Integrador *Centinela*: **Rama A** (clasificación de imágenes con CNN / *transfer learning*), **Rama B** (series temporales con RNN/LSTM/GRU) y **Rama C** (fusión multimodal). Trae lista la mecánica de soporte —generación de datos sintéticos, ventaneo, *DataLoaders* y una función de fusión de ejemplo— y deja marcadas con `# TODO (estudiante)` las piezas evaluables: la **arquitectura de cada modelo**, los **bucles de entrenamiento**, **Grad-CAM** y la **cabeza de fusión**.

> **Cómo se usa:** se clona este cuaderno, se ejecutan las celdas de soporte en orden y se completan las celdas marcadas `# TODO (estudiante)`. En Google Colab: *Entorno de ejecución → Ejecutar todas*. Las celdas de soporte corren de punta a punta; las celdas `# TODO` traen *stubs* que **no rompen la ejecución** (imprimen un recordatorio) hasta que el estudiante escribe su solución.

> ⚠️ **Andamiaje — NO es la solución.** Este cuaderno **no** contiene ninguna CNN, RNN ni cabeza de fusión entrenadas, y usa **datos sintéticos** de juguete (formas geométricas y una serie generada), **no** los datos reales del proyecto. Su propósito es enseñar la *mecánica* de las tres arquitecturas; el diseño, el entrenamiento y el análisis los aporta el estudiante sobre **sus propios datos**.

---
**Contenido**
0. Preparación del entorno (semilla, *device*, paleta USTA, política de datos)
1. **Rama A — CNN (imágenes):** datos sintéticos de formas · `DataLoader` provisto · `# TODO` modelo / *transfer learning* / *training loop* / Grad-CAM
2. **Rama B — RNN/LSTM (series):** serie sintética · ventaneo provisto · `# TODO` arquitectura recurrente / *training loop*
3. **Rama C — Fusión multimodal:** función de fusión de ejemplo provista · `# TODO` cabeza de fusión / entrenamiento conjunto · *caveat* de co-registro y *leakage*
4. Checklist de entrega + rúbrica unificada de la Fase 2


## 0. Preparación del entorno

Se importan las librerías de soporte (`numpy`, `scikit-learn`, `matplotlib`) y, **si está disponible**, `torch`. Se fija una **semilla** global para reproducibilidad y se detecta el `device` (CPU o GPU). La separación de miles se mostrará con punto, según la convención del material del curso.

> 💡 *PyTorch y torchvision como soporte:* a diferencia del andamiaje de la Fase 1 (donde `torch` era enteramente un TODO), aquí los `DataLoader` y tensores de ejemplo usan PyTorch. La importación se hace de forma **tolerante**: si `torch` aún no estuviera instalado, el cuaderno **no se rompe** y las celdas de soporte que no lo necesitan siguen corriendo. `torchvision` es **opcional**.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Celda de soporte 1 · Preparacion del entorno
# Qué hace · Importa librerias de soporte, fija la semilla global, detecta el device
#            y configura el estilo de las graficas con la paleta USTA.
# Por qué  · Una semilla fija hace reproducibles los resultados; detectar el device
#            permite el mismo codigo en CPU (demo) y en GPU (datos reales en Colab).

# En Colab estas librerias ya vienen instaladas. Si faltara alguna, descomentar:
# !pip -q install scikit-learn matplotlib torch torchvision

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Reproducibilidad: una misma semilla produce los mismos resultados en cada ejecucion.
SEMILLA = 42
np.random.seed(SEMILLA)

# Importacion TOLERANTE de PyTorch: si no estuviera instalado, el cuaderno no se rompe.
# (torchvision es OPCIONAL: solo se necesita para transfer learning con pesos preentrenados.)
try:
    import torch
    from torch.utils.data import TensorDataset, DataLoader
    torch.manual_seed(SEMILLA)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    TORCH_OK = True
except Exception as e:  # pragma: no cover  (entorno sin torch)
    torch = None
    TensorDataset = DataLoader = None
    DEVICE = "cpu"
    TORCH_OK = False
    print("Aviso: PyTorch no esta disponible en este entorno ->", e)
    print("Las celdas de soporte que no usan torch seguiran funcionando.")

try:
    import torchvision  # OPCIONAL (pesos preentrenados para transfer learning)
    TORCHVISION_OK = True
except Exception:
    torchvision = None
    TORCHVISION_OK = False

# Paleta USTA para las graficas
USTA_MORADO, USTA_ROSA, USTA_NAVY = "#3D008D", "#ED1E79", "#001A4D"
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

# Separador de miles con punto, segun la convencion del material del curso.
def miles(n):
    """Formatea un entero con punto como separador de miles (p. ej. 1.234.567)."""
    return f"{int(n):,}".replace(",", ".")

print("Entorno de soporte listo · numpy", np.__version__)
print(f"PyTorch disponible: {TORCH_OK}   ·   torchvision disponible: {TORCHVISION_OK}   ·   device: {DEVICE}")


> [!warning] Política de datos del Proyecto Centinela
> Este andamiaje usa **datos SINTÉTICOS de demostración** (formas geométricas y una serie generada) **solo** para enseñar la mecánica. La entrega evaluable de la Fase 2 se construye sobre **datos REALES elegidos por el estudiante** y aprobados (visión + serie temporal del mismo problema/cliente de la Fase 1). **No se admiten datasets de Kaggle como eje del proyecto**: el dato debe provenir de una fuente declarada y justificada (p. ej. recolección propia, IDEAM/NASA POWER para clima, repositorios públicos institucionales). Reutilizar estos datos de juguete **no satisface** la actividad.


## 1. Rama A — CNN (imágenes)

Para practicar la mecánica de visión **sin adelantar nada del proyecto**, se generan **imágenes sintéticas de juguete**: formas geométricas (círculo, cuadrado, triángulo) dibujadas con `numpy` sobre fondos ruidosos, en **3 clases**. Esto sustituye, en la demostración, a un `ImageFolder` con fotos reales.

> ☕ **Puente con el proyecto:** en el proyecto real estas imágenes serán las del problema elegido (hojas de café con/sin plaga, especies de fauna, patologías, vegetación satelital), cargadas típicamente con `torchvision.datasets.ImageFolder`. El andamiaje del `DataLoader` y del bucle es el mismo; solo cambia la fuente de los píxeles.

Se proveen, **listos para usar**: (1) el generador de imágenes sintéticas, (2) un *helper* que las empaqueta en `DataLoader` de PyTorch y (3) un placeholder comentado de cómo se cargaría un `ImageFolder` real. Se dejan como `# TODO (estudiante)`: la **definición del modelo CNN / transfer learning**, el **bucle de entrenamiento** y **Grad-CAM**.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Celda de soporte 2 · Imagenes sinteticas de juguete (FUNCIONAL)
# Qué hace · Genera imagenes 1xHxW con una forma geometrica por clase (circulo/cuadrado/triangulo)
#            sobre fondo ruidoso, y devuelve arreglos numpy (X: N,1,H,W ; y: N,).
# Por qué  · Permite ejercitar toda la tuberia de vision (dataloader, train loop, Grad-CAM)
#            con computo minimo y sin descargar ningun dataset.

IMG = 28               # lado de la imagen (px)
CLASES_A = ("circulo", "cuadrado", "triangulo")
N_POR_CLASE = 60       # pequeño a proposito: computo minimo

def _lienzo_ruidoso(rng):
    """Fondo gris con ruido leve, en [0, 1]."""
    return (0.15 + 0.10 * rng.standard_normal((IMG, IMG))).clip(0, 1)

def _dibujar_circulo(img, rng):
    cy, cx = rng.integers(8, IMG - 8, size=2)
    r = rng.integers(5, 9)
    yy, xx = np.ogrid[:IMG, :IMG]
    img[(yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2] = 1.0
    return img

def _dibujar_cuadrado(img, rng):
    lado = rng.integers(9, 14)
    y0 = rng.integers(2, IMG - lado - 2); x0 = rng.integers(2, IMG - lado - 2)
    img[y0:y0 + lado, x0:x0 + lado] = 1.0
    return img

def _dibujar_triangulo(img, rng):
    h = rng.integers(10, 15)
    y0 = rng.integers(2, IMG - h - 2); x0 = rng.integers(2, IMG - h - 2)
    for k in range(h):                       # triangulo rectangulo simple
        img[y0 + k, x0:x0 + k + 1] = 1.0
    return img

def generar_imagenes_sinteticas(n_por_clase=N_POR_CLASE, semilla=SEMILLA):
    """Devuelve (X, y) con X de forma (N, 1, IMG, IMG) en float32 y y en {0,1,2}."""
    rng = np.random.default_rng(semilla)
    dibujantes = (_dibujar_circulo, _dibujar_cuadrado, _dibujar_triangulo)
    X, y = [], []
    for clase, dibujar in enumerate(dibujantes):
        for _ in range(n_por_clase):
            img = dibujar(_lienzo_ruidoso(rng), rng)
            X.append(img[None, :, :])        # canal 1 -> (1, H, W)
            y.append(clase)
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)
    # mezclar
    perm = rng.permutation(len(y))
    return X[perm], y[perm]

X_img, y_img = generar_imagenes_sinteticas()
print(f"Imagenes sinteticas: {miles(len(X_img))} muestras · {len(CLASES_A)} clases {CLASES_A}")
print(f"Forma del tensor de imagenes (N, C, H, W): {X_img.shape}")

# Vista rapida: una muestra por clase
fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.8))
for ax, clase in zip(axes, range(3)):
    idx = int(np.where(y_img == clase)[0][0])
    ax.imshow(X_img[idx, 0], cmap="Purples"); ax.set_title(CLASES_A[clase]); ax.axis("off")
plt.suptitle("Rama A · muestras sinteticas (una por clase)"); plt.tight_layout(); plt.show()


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Celda de soporte 3 · DataLoaders de imagenes (FUNCIONAL si hay torch)
# Qué hace · Parte (70/15/15) y empaqueta las imagenes sinteticas en DataLoaders de PyTorch.
# Por qué  · El bucle de entrenamiento del estudiante (TODO) consumira estos DataLoaders;
#            asi la mecanica de iteracion por lotes queda resuelta.

def particion_estratificada(X, y, semilla=SEMILLA):
    """Divide en 70% train / 15% val / 15% test respetando el balance de clases."""
    X_tmp, X_te, y_tmp, y_te = train_test_split(
        X, y, test_size=0.15, random_state=semilla, stratify=y)
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_tmp, y_tmp, test_size=0.1765, random_state=semilla, stratify=y_tmp)  # ~0.15 del total
    return (X_tr, y_tr), (X_va, y_va), (X_te, y_te)

def construir_dataloaders_imagen(X, y, batch_size=16, semilla=SEMILLA):
    """Devuelve dict de DataLoaders {train, val, test} o None si torch no esta disponible."""
    (Xtr, ytr), (Xva, yva), (Xte, yte) = particion_estratificada(X, y, semilla)
    print(f"Particion imagenes -> train {miles(len(ytr))} · val {miles(len(yva))} · test {miles(len(yte))}")
    if not TORCH_OK:
        print("torch no disponible: se omite la creacion de DataLoaders (no rompe el cuaderno).")
        return None
    def _dl(Xa, ya, shuffle):
        ds = TensorDataset(torch.tensor(Xa), torch.tensor(ya))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)
    return {"train": _dl(Xtr, ytr, True),
            "val":   _dl(Xva, yva, False),
            "test":  _dl(Xte, yte, False)}

dls_img = construir_dataloaders_imagen(X_img, y_img, batch_size=16)
if dls_img is not None:
    xb, yb = next(iter(dls_img["train"]))
    print(f"Lote de imagenes -> X {tuple(xb.shape)}  ·  y {tuple(yb.shape)}  ·  dtype {xb.dtype}")

# --- PLACEHOLDER (comentado) · asi se cargaria un ImageFolder REAL en el proyecto ---
# from torchvision import datasets, transforms
# tfm = transforms.Compose([
#     transforms.Resize((224, 224)),                 # tamaño esperado por ResNet/EfficientNet
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
# ])
# ds_real = datasets.ImageFolder("ruta/a/mis/imagenes", transform=tfm)
# dl_real = DataLoader(ds_real, batch_size=32, shuffle=True)


### `# TODO (estudiante)` · Modelo CNN / *transfer learning*, *training loop* y Grad-CAM

> [!todo] Tarea del estudiante (Rama A)
> Aquí está el trabajo evaluable de visión (criterio 1 de la rúbrica). Sobre **datos reales**, no sobre estas formas de juguete, se debe:
>
> 1. **Definir la arquitectura.** Comparar al menos **dos** redes preentrenadas (p. ej. `ResNet50`, `VGG16`, `EfficientNet` vía `torchvision.models`) y decidir entre **Feature Extraction** (congelar pesos) y **Fine-Tuning** (reentrenar capas superiores), justificándolo por tamaño y similitud del dataset con ImageNet.
> 2. **Adaptar la cabeza** al número de clases del problema y aplicar **data augmentation** (rotación, *flip*, *zoom*, brillo).
> 3. **Escribir el bucle de entrenamiento** con monitoreo de pérdida/accuracy en validación, *early stopping* y *checkpointing* a Drive (la rama visual puede no caber en una sola sesión de Colab).
> 4. **Grad-CAM:** visualizar los mapas de activación para interpretar qué aprende la red.
>
> 💡 *Las imágenes de juguete son 1×28×28.* Para transfer learning con pesos de ImageNet habrá que llevar las imágenes reales a **3×224×224** y normalizar con las medias/desviaciones de ImageNet (ver el placeholder de la celda anterior).

> ⚠️ La celda siguiente trae **stubs** (`modelo_A = None`, `print(...)`) para que el cuaderno **corra de punta a punta**. El estudiante reemplaza los stubs por su implementación.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TODO (estudiante) · Rama A: CNN / transfer learning, training loop y Grad-CAM
# Reemplazar los stubs de abajo por la implementacion propia (sobre datos REALES).
# Los stubs NO rompen la ejecucion: dejan variables placeholder e imprimen un recordatorio.

# 1) Modelo CNN / transfer learning ------------------------------------------
# Pista (descomentar y completar cuando torchvision este disponible):
# from torchvision import models
# import torch.nn as nn
# base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)   # o resnet50 / efficientnet_b0
# for p in base.parameters():            # Feature Extraction: congelar pesos
#     p.requires_grad = False
# base.fc = nn.Linear(base.fc.in_features, len(CLASES_A))            # adaptar la cabeza
# modelo_A = base.to(DEVICE)
modelo_A = None  # TODO (estudiante): construir la CNN / red preentrenada adaptada

# 2) Bucle de entrenamiento ---------------------------------------------------
def entrenar_cnn(modelo, dataloaders, epocas=10, lr=1e-3):
    """TODO (estudiante): definir perdida (CrossEntropyLoss) y optimizador (Adam),
    iterar epocas sobre dataloaders['train'] con los 5 pasos
    (zero_grad -> forward -> loss -> backward -> step), validar en dataloaders['val'],
    aplicar early stopping y devolver el historial de metricas."""
    print("TODO: implementar el bucle de entrenamiento de la CNN (criterio 1 de la rubrica).")
    return {"train_loss": [], "val_loss": [], "val_acc": []}

# 3) Grad-CAM -----------------------------------------------------------------
def grad_cam(modelo, imagen):
    """TODO (estudiante): registrar hooks sobre la ultima capa convolucional,
    propagar la imagen, ponderar los mapas de activacion por el gradiente de la clase
    y devolver el mapa de calor para superponerlo sobre la imagen."""
    print("TODO: implementar Grad-CAM para interpretar las activaciones de la CNN.")
    return None

# Llamadas de demostracion (NO entrenan: solo confirman que el andamiaje no se rompe)
_hist_A = entrenar_cnn(modelo_A, dls_img, epocas=1)
_cam = grad_cam(modelo_A, X_img[0])
print("Rama A · andamiaje listo. modelo_A =", modelo_A, "(stub: el estudiante lo implementa).")


## 2. Rama B — RNN/LSTM (series temporales)

Para la rama secuencial se genera una **serie temporal sintética** con tendencia, estacionalidad y ruido (un sustituto de juguete de, p. ej., precipitación mensual en zona cafetera o demanda horaria). Se provee, **listo para usar**, el *helper* de **ventaneo deslizante** (*sliding window*) que transforma la serie 1-D en pares `(ventana, siguiente_valor)` y, si hay torch, sus `DataLoader`.

Se deja como `# TODO (estudiante)`: la **arquitectura recurrente** (RNN simple de contraste + LSTM + GRU) y el **bucle de entrenamiento**.

> ☕ **Puente con el proyecto:** en el proyecto real la serie es la variable objetivo elegida (precipitación, transacciones, demanda) con mínimo 1.000 observaciones, respetando el **orden temporal** en la partición (nunca barajar antes de ventanear).


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Celda de soporte 4 · Serie temporal sintetica + ventaneo (FUNCIONAL)
# Qué hace · Genera una serie 1-D (tendencia + estacionalidad + ruido), la normaliza,
#            y la convierte en ventanas deslizantes (lookback -> siguiente valor).
# Por qué  · El ventaneo y la particion temporal son la mecanica que el modelo recurrente
#            (TODO) consumira; aqui quedan resueltos con computo minimo.

N_PASOS = 600          # longitud de la serie sintetica
LOOKBACK = 14          # tamaño de ventana por defecto (la actividad sugiere 14)

def generar_serie_sintetica(n=N_PASOS, semilla=SEMILLA):
    """Serie 1-D = tendencia suave + estacionalidad + ruido gaussiano."""
    rng = np.random.default_rng(semilla)
    t = np.arange(n)
    tendencia = 0.002 * t
    estacional = 0.8 * np.sin(2 * np.pi * t / 30) + 0.3 * np.sin(2 * np.pi * t / 7)
    ruido = 0.15 * rng.standard_normal(n)
    return (tendencia + estacional + ruido).astype(np.float32)

def ventanear(serie, lookback=LOOKBACK):
    """Convierte una serie 1-D en (X, y): X de forma (M, lookback, 1) y y (M, 1).

    Cada muestra usa 'lookback' pasos consecutivos para predecir el siguiente valor.
    El eje final =1 es la dimension de 'features' que esperan las capas recurrentes
    de PyTorch (batch, timesteps, features)."""
    X, y = [], []
    for i in range(len(serie) - lookback):
        X.append(serie[i:i + lookback])
        y.append(serie[i + lookback])
    X = np.asarray(X, dtype=np.float32)[..., None]   # (M, lookback, 1)
    y = np.asarray(y, dtype=np.float32)[:, None]     # (M, 1)
    return X, y

def particion_temporal(X, y, frac_train=0.70, frac_val=0.15):
    """Particion que RESPETA el orden temporal (sin barajar): train | val | test."""
    n = len(X)
    i_tr = int(n * frac_train)
    i_va = int(n * (frac_train + frac_val))
    return (X[:i_tr], y[:i_tr]), (X[i_tr:i_va], y[i_tr:i_va]), (X[i_va:], y[i_va:])

serie = generar_serie_sintetica()
# Normalizacion simple (media/desv de la porcion de entrenamiento, para evitar leakage).
corte_tr = int(len(serie) * 0.70)
mu, sigma = serie[:corte_tr].mean(), serie[:corte_tr].std() + 1e-8
serie_norm = (serie - mu) / sigma

X_seq, y_seq = ventanear(serie_norm, lookback=LOOKBACK)
(Xtr_s, ytr_s), (Xva_s, yva_s), (Xte_s, yte_s) = particion_temporal(X_seq, y_seq)
print(f"Serie sintetica: {miles(len(serie))} pasos · lookback {LOOKBACK}")
print(f"Ventanas -> total {miles(len(X_seq))} · forma X {X_seq.shape} (M, timesteps, features)")
print(f"Particion temporal -> train {miles(len(Xtr_s))} · val {miles(len(Xva_s))} · test {miles(len(Xte_s))}")

# DataLoaders (solo si torch esta disponible)
dls_seq = None
if TORCH_OK:
    def _dl_seq(Xa, ya, shuffle):
        ds = TensorDataset(torch.tensor(Xa), torch.tensor(ya))
        return DataLoader(ds, batch_size=32, shuffle=shuffle)
    dls_seq = {"train": _dl_seq(Xtr_s, ytr_s, False),   # series: NO barajar
               "val":   _dl_seq(Xva_s, yva_s, False),
               "test":  _dl_seq(Xte_s, yte_s, False)}
    xb, yb = next(iter(dls_seq["train"]))
    print(f"Lote de secuencias -> X {tuple(xb.shape)}  ·  y {tuple(yb.shape)}")

# Vista rapida de la serie
fig, ax = plt.subplots(figsize=(7.5, 3))
ax.plot(serie, color=USTA_MORADO, lw=1.2)
ax.axvline(corte_tr, color=USTA_ROSA, ls="--", lw=1, label="fin entrenamiento")
ax.set_title("Rama B · serie temporal sintetica"); ax.set_xlabel("paso"); ax.legend()
plt.tight_layout(); plt.show()


### `# TODO (estudiante)` · Arquitectura recurrente y *training loop*

> [!todo] Tarea del estudiante (Rama B)
> Aquí está el trabajo evaluable secuencial (criterio 2 de la rúbrica). Sobre la **serie real**, no sobre esta sintética, se debe:
>
> 1. **Implementar obligatoriamente** dos celdas con compuertas: **LSTM** y **GRU** (1–2 capas, `hidden` 64–128, `dropout` entre capas), e incluir una **RNN simple (Vanilla)** **solo como contraste** para evidenciar el desvanecimiento del gradiente.
> 2. **Escribir el bucle de entrenamiento** con `MSELoss` (o MAE), optimizador `Adam` (`lr≈1e-3`), monitoreo en validación y *early stopping*. Recordar la forma de entrada `(batch, timesteps, features)`.
> 3. **Comparar** las tres arquitecturas con RMSE/MAE/MAPE y documentar el desvanecimiento del gradiente (traza de normas **o** diferencia de desempeño en dependencias largas).
>
> 💡 En PyTorch, `nn.LSTM(input_size=1, hidden_size=64, batch_first=True)` devuelve `(salidas, (h_n, c_n))`; suele tomarse el **último** paso temporal (`salidas[:, -1, :]`) hacia una capa densa de regresión.

> ⚠️ La celda siguiente trae **stubs** que no rompen la ejecución. El estudiante reemplaza `modelo_B = None` y el cuerpo de `entrenar_rnn(...)`.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TODO (estudiante) · Rama B: arquitectura recurrente y bucle de entrenamiento
# Reemplazar los stubs por la implementacion propia (LSTM y GRU + RNN simple de contraste).
# Los stubs NO rompen la ejecucion.

# 1) Arquitectura recurrente --------------------------------------------------
# Pista (descomentar y completar cuando torch este disponible):
# import torch.nn as nn
# class RedRecurrente(nn.Module):
#     def __init__(self, tipo="lstm", n_features=1, hidden=64, capas=1, p_dropout=0.2):
#         super().__init__()
#         celda = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[tipo]
#         self.rec = celda(n_features, hidden, num_layers=capas,
#                          batch_first=True, dropout=p_dropout if capas > 1 else 0.0)
#         self.cabeza = nn.Linear(hidden, 1)
#     def forward(self, x):
#         salidas, _ = self.rec(x)            # (batch, timesteps, hidden)
#         return self.cabeza(salidas[:, -1])  # ultimo paso -> regresion
# modelo_B = RedRecurrente(tipo="lstm").to(DEVICE)
modelo_B = None  # TODO (estudiante): definir LSTM/GRU (y RNN simple de contraste)

# 2) Bucle de entrenamiento ---------------------------------------------------
def entrenar_rnn(modelo, dataloaders, epocas=20, lr=1e-3):
    """TODO (estudiante): definir MSELoss y Adam, iterar epocas sobre
    dataloaders['train'] con los 5 pasos, validar en dataloaders['val'],
    aplicar early stopping y devolver el historial. Comparar RNN vs LSTM vs GRU."""
    print("TODO: implementar el bucle de entrenamiento recurrente (criterio 2 de la rubrica).")
    return {"train_loss": [], "val_loss": []}

# Llamada de demostracion (NO entrena: confirma que el andamiaje no se rompe)
_hist_B = entrenar_rnn(modelo_B, dls_seq, epocas=1)
print("Rama B · andamiaje listo. modelo_B =", modelo_B, "(stub: el estudiante lo implementa).")


## 3. Rama C — Fusión multimodal

La fusión integra las ramas A y B en un **único predictor**: cada rama produce un **vector de representación** (*embedding*) y la fusión los **concatena** para decidir usando, a la vez, lo que el sistema *ve* (imagen) y lo que *recuerda* (serie):

$$ \hat{y} = \sigma\!\left(W_f\,[\,z_{img} \,\Vert\, z_{seq}\,] + b_f\right) $$

Se provee, **listo para usar**, una función de fusión **de ejemplo** que muestra cómo concatenar un vector de imagen con un vector temporal. Se deja como `# TODO (estudiante)`: la **cabeza de fusión** (red densa de decisión) y el **entrenamiento conjunto**, además del **estudio de ablación** (solo-imagen / solo-serie / fusionado).


> [!warning] Co-registro y fuga de datos (*leakage*) — la decisión más importante de la Rama C
> La fusión solo es legítima si existe una **clave de alineación** real que une cada imagen con su serie: misma entidad y misma ventana espaciotemporal. El **ideal** es el co-registro *por muestra*, con unidad **(finca, semana)** cuando hay datos georreferenciados y fechados.
>
> En el escenario de referencia *"Centinela del Café"*, la rama de imagen (RoCoLe) y la serie climática (NASA POWER / IDEAM) **solo se alinean a nivel agregado** (región/periodo), **no por muestra**: las imágenes **no traen fecha ni GPS**. Emparejar artificialmente cada imagen con una ventana climática, o adjuntar una *feature* climática **constante** a todas las muestras, **es fuga de datos** y debe declararse y evitarse.
> - **Aceptable:** alineación **agregada** (región/periodo), siempre que se **declare con sus limitaciones**.
> - **Ideal (Estratégico):** **co-registro genuino por muestra**, demostrando la clave (qué campo/fecha/id une las modalidades).
>
> 👉 En esta demostración los *embeddings* son **vectores aleatorios independientes**: NO representan una alineación real. Sirven solo para enseñar la mecánica de la concatenación.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Celda de soporte 5 · Fusion de ejemplo (FUNCIONAL)
# Qué hace · Muestra como concatenar un embedding de imagen (z_img) con uno temporal (z_seq)
#            en un unico vector multimodal listo para la cabeza de fusion.
# Por qué  · Aisla la MECANICA de la concatenacion; el estudiante luego conecta los
#            embeddings REALES (penultima capa de la CNN y de la RNN) y entrena la cabeza.

DIM_IMG, DIM_SEQ = 64, 32   # dimensiones de ejemplo de cada embedding

def fusionar_embeddings(z_img, z_seq):
    """Concatena dos embeddings por muestra -> vector multimodal (N, DIM_IMG + DIM_SEQ).

    Parametros
    ----------
    z_img : np.ndarray (N, DIM_IMG)  · representacion de la rama A (imagen)
    z_seq : np.ndarray (N, DIM_SEQ)  · representacion de la rama B (serie)

    Requisito CLAVE: ambos arreglos deben estar CO-REGISTRADOS (la fila i de z_img
    y la fila i de z_seq describen la MISMA unidad finca-semana). Ver la advertencia
    de leakage de la celda anterior.
    """
    z_img = np.asarray(z_img, dtype=np.float32)
    z_seq = np.asarray(z_seq, dtype=np.float32)
    if z_img.shape[0] != z_seq.shape[0]:
        raise ValueError(
            f"Numero de muestras distinto ({z_img.shape[0]} vs {z_seq.shape[0]}): "
            "los embeddings no estan co-registrados muestra a muestra.")
    return np.concatenate([z_img, z_seq], axis=1)

# Demostracion con embeddings ALEATORIOS (placeholder: NO es una alineacion real)
rng = np.random.default_rng(SEMILLA)
N_DEMO = 50
z_img_demo = rng.standard_normal((N_DEMO, DIM_IMG)).astype(np.float32)
z_seq_demo = rng.standard_normal((N_DEMO, DIM_SEQ)).astype(np.float32)
z_fusion = fusionar_embeddings(z_img_demo, z_seq_demo)
print(f"z_img {z_img_demo.shape}  +  z_seq {z_seq_demo.shape}  ->  z_fusion {z_fusion.shape}")
print("Recordatorio: estos embeddings son aleatorios e independientes (no co-registrados).")


### `# TODO (estudiante)` · Cabeza de fusión y entrenamiento conjunto

> [!todo] Tarea del estudiante (Rama C)
> Aquí está el trabajo evaluable de fusión (criterio 3 de la rúbrica, **el de mayor peso: 30 %**). Se debe:
>
> 1. **Extraer embeddings reales:** reutilizar la CNN (Rama A) y la RNN (Rama B) ya entrenadas como **extractores** —tomar la penúltima capa de cada una— sobre datos **co-registrados** (misma unidad finca-semana).
> 2. **Definir la cabeza de fusión:** una red densa que reciba `[z_img ‖ z_seq]` y produzca la decisión final (clasificación o regresión, según el problema).
> 3. **Entrenamiento conjunto:** entrenar la cabeza (y, opcionalmente, *fine-tuning* parcial de las ramas) con su propio bucle.
> 4. **Estudio de ablación:** comparar **solo-imagen**, **solo-serie** y **fusionado**; responder *cuánto* y *cuándo* aporta cada modalidad (un resultado negativo bien analizado es válido).
> 5. **Declarar la alineación:** documentar la clave de co-registro y, si es agregada, sus limitaciones (ver la advertencia de *leakage*).

> ⚠️ La celda siguiente trae **stubs** que no rompen la ejecución. El estudiante reemplaza `cabeza_fusion = None` y el cuerpo de `entrenar_fusion(...)`.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TODO (estudiante) · Rama C: cabeza de fusion y entrenamiento conjunto
# Reemplazar los stubs por la implementacion propia. Los stubs NO rompen la ejecucion.

# 1) Cabeza de fusion (red densa de decision) ---------------------------------
# Pista (descomentar y completar cuando torch este disponible):
# import torch.nn as nn
# class CabezaFusion(nn.Module):
#     def __init__(self, dim_in=DIM_IMG + DIM_SEQ, n_salidas=len(CLASES_A), oculta=64):
#         super().__init__()
#         self.red = nn.Sequential(
#             nn.Linear(dim_in, oculta), nn.ReLU(), nn.Dropout(0.3),
#             nn.Linear(oculta, n_salidas))
#     def forward(self, z):       # z = [z_img || z_seq] co-registrado
#         return self.red(z)
# cabeza_fusion = CabezaFusion().to(DEVICE)
cabeza_fusion = None  # TODO (estudiante): definir la cabeza densa de fusion

# 2) Entrenamiento conjunto + ablacion ----------------------------------------
def entrenar_fusion(cabeza, z_fusion, etiquetas, epocas=15, lr=1e-3):
    """TODO (estudiante): convertir z_fusion/etiquetas a tensores, definir perdida
    y optimizador, entrenar la cabeza con los 5 pasos y devolver el historial.
    Repetir el experimento en modo SOLO-IMAGEN, SOLO-SERIE y FUSIONADO (ablacion)."""
    print("TODO: implementar el entrenamiento conjunto de la cabeza de fusion (criterio 3).")
    return {"train_loss": []}

# Llamada de demostracion (NO entrena: confirma que el andamiaje no se rompe)
_hist_C = entrenar_fusion(cabeza_fusion, z_fusion, etiquetas=None, epocas=1)
print("Rama C · andamiaje listo. cabeza_fusion =", cabeza_fusion, "(stub: el estudiante la implementa).")


## 4. Checklist de entrega + rúbrica

Una vez completadas las tres ramas sobre los **datos reales** del proyecto (no sobre estos datos sintéticos), la Fase 2 se entrega con estas piezas. La calificación se rige por la **rúbrica unificada** (abajo), cuyos cinco pesos suman 100 %.

**Checklist de entrega**

- [ ] **Notebook** (`.ipynb`) ejecutable y documentado con las tres ramas: CNN con *transfer learning* y Grad-CAM (A), LSTM + GRU + RNN de contraste (B), cabeza de fusión con estudio de ablación (C).
- [ ] **Pesos entrenados** (`.h5`/`.pth`) de los modelos, con *checkpoints* a Drive documentados.
- [ ] **Informe técnico** (PDF) y **presentación ejecutiva** para audiencia no técnica.
- [ ] **Declaración de alineación** entre modalidades (clave de co-registro o, si es agregada, sus limitaciones).
- [ ] **Bitácora de IA**: qué se pidió, qué devolvió la IA y qué se **aceptó / modificó / rechazó** y por qué.
- [ ] **Auditoría a la IA**: al menos **un caso real** en que la IA falló (p. ej. dimensiones incompatibles, fuga de datos, métrica mal interpretada) y cómo se corrigió.
- [ ] Carpeta comprimida (`.zip`) con todo lo anterior.

**Rúbrica unificada de la Fase 2** (autoritativa; los pesos suman 100 %):

| Criterio (peso) | Qué evidencia |
|---|---|
| **1. Rama visual — CNN y Transfer Learning (20 %)** | Compara 2 arquitecturas preentrenadas (parámetros, profundidad, inferencia); justifica *Feature Extraction* vs *Fine-Tuning*; aporta **Grad-CAM** y evaluación por clase. |
| **2. Rama temporal — LSTM/GRU + RNN de contraste (20 %)** | LSTM y GRU correctos + RNN simple de contraste; respeta el **orden temporal**; evidencia el desvanecimiento del gradiente; compara con RMSE/MAE/MAPE. |
| **3. Fusión, ablación y alineación legítima (30 %)** | Demuestra la **clave de alineación** entre modalidades; fusión funcional con **ablación** (solo-imagen / solo-serie / fusionado) y análisis honesto, evitando *leakage*. |
| **4. Uso responsable de IA (15 %)** | **Bitácora** con decisiones deliberadas; **sustentación** oral de cada decisión; **auditoría** con un fallo no trivial corregido. |
| **5. Comunicación de resultados (15 %)** | Visualizaciones estáticas claras (dashboard interactivo como *bonus*); informe con narrativa, recomendaciones accionables y citación APA. |

**Total de la Fase 2: 30 % del curso · 46 h.**

> [!warning] Este andamiaje no se entrega tal cual
> Este cuaderno es punto de partida y trabaja con **datos sintéticos**. La entrega evaluable debe aplicarse a los **datos reales** del problema elegido (visión + serie del mismo cliente de la Fase 1), e incluir los pesos, el informe, la bitácora y la auditoría de IA. Reutilizar este scaffold sin adaptarlo **no satisface** la actividad.

### 🔗 Relacionado
- Página principal de la bóveda y mapa del Segundo Cerebro.
- Syllabus del Curso y Ruta de Aprendizaje.
- Documento maestro: **Proyecto Integrador Centinela** (especificación longitudinal de las tres fases).
- **Fase 2 (evaluable):** Actividad 2 · "Sistema Multimodal Especializado" (ramas A, B y C).
- **Base teórica:** Módulo 2 — Capítulos 1 (CNN), 2 (LSTM/GRU) y 3 (modelos generativos).
- **Fases vecinas:** Fase 1 (línea base con MLP) · Fase 3 (pipeline y despliegue en borde).
- **Uso de IA:** plantilla de la Bitácora de IA para registrar el proceso de la fase.
